# agent_tools

> Learning operations that a host agent wraps as tools.

#| export
A host agent, such as Leela, makes a walk, reads its state, grades an answer, notes a confusion, changes a walk, adds a source, restores an old version, explains the step on screen and knows which page the learner sees. This module holds those operations. It imports from `learner`, `models`, `sources`, `tutor`, `path` and `show`. Each operation takes the learner's store `lr`, and the host's model call `ask(prompt, sp='')` when it needs a model.

**The contract of every operation**

- An operation returns one dict with these fields: `status` (`'ok'`, `'error'` or `'stopped'`), `operation_id`, `vault`, `walk`, `step`, `revision`, `data` (a dict) and `error`. `result` builds it.
- `error` is `None`, or one of `ERRORS`: `'vault_locked'`, `'read_only'`, `'source_missing'`, `'model_refused'`, `'privacy_refused'`, `'stopped'`, `'not_found'`, `'attempt_conflict'`, `'stale_state'`. A host adds its own names for one call with `extra`: `failed('thread_busy', extra=('thread_busy',))`.
- The host owns privacy. `create_walk` takes `guard(text) -> text` from the host and runs no engine guard of its own. `record_attempt` takes the `revision` that the caller last saw and refuses an old one with `'stale_state'`.
- An expected failure never raises. It has `status` `'error'` and an `error`, and `data['message']` says why. A stop has `status` `'stopped'` and `error` `'stopped'`.
- The values are JSON values only. The caller owns `operation_id`: the engine leaves it `None`. `vault` is the file name of the vault. `revision` is the `updated` time of the walk.
- Stop is a `stopping()` callback that returns true after a stop. The operation checks it before each model call and after each one, so before it saves anything. A stop leaves no partial walk.
- `TOOLS` lists each operation that a host makes a tool, with the arguments that the model gives. The host gives `lr`, `ask`, `stopping` and `progress` itself, and for `explain_here` also `jid` and `i`: the step on the page that the learner sees. `READS` names the tools that change nothing.
- A change to a walk takes the `revision` that the caller last saw, and refuses an old one with `'stale_state'`. Each change is one change with an Undo in the history of the walk, so `list_versions` and `restore_version` see it.

In [ ]:
#| default_exp agent_tools

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # nor the real ramabana config
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| hide
from fastcore.test import *

Import what this module uses. The model, the walk and the tutor come from the engine modules above it.

In [ ]:
#| export
import re, time
from fastcore.utils import *
from abhyasa import learner, path, show
from abhyasa.learner import Learning, teach_sp
from abhyasa.models import Withheld
from abhyasa.sources import VaultSource, clip, fence
from urllib.parse import unquote, urlsplit
from abhyasa.tutor import Tutor

## The result

`result` builds the one shape. `failed` builds an expected failure: a stop has the status `'stopped'`, and each other failure has the status `'error'`.

In [ ]:
#| export
ERRORS = ('vault_locked', 'read_only', 'source_missing', 'model_refused', 'privacy_refused', 'stopped', 'not_found', 'attempt_conflict', 'stale_state')
STATUSES = ('ok', 'error', 'stopped')
FIELDS = ('operation_id', 'vault', 'walk', 'step', 'revision', 'data', 'error')

def result(
    status:str,   # one of `STATUSES`
    extra:tuple=(),   # error names that the host adds to `ERRORS`, for this call only
    **kw          # a value for any field of `FIELDS`
) -> dict:
    "The one result shape: `status` and each field of `FIELDS`, empty unless `kw` gives it."
    if status not in STATUSES: raise ValueError(f'no status {status!r}')
    if (bad := set(kw) - set(FIELDS)): raise TypeError(f'no result field {sorted(bad)}')
    if kw.get('error') not in (None, *ERRORS, *extra): raise ValueError(f"no error {kw['error']!r}")
    return dict(status=status, operation_id=None, vault=None, walk=None, step=None, revision=None, data={}, error=None) | kw

def failed(
    error:str,       # one of `ERRORS`
    message:str='',  # why, in plain words; it goes in `data['message']`
    extra:tuple=(),  # error names that the host adds to `ERRORS`, for this call only
    **kw             # a value for `walk`, `step` or another field
) -> dict:
    "An expected failure as a result."
    return result('stopped' if error == 'stopped' else 'error', extra, error=error, data=dict(message=message), **kw)

`TOOLS` names each operation that a host makes a tool. The arguments are the ones that the model gives. They have the names of the function parameters, so a host passes them straight through.

In [ ]:
#| export
TOOLS = (
    ('create_walk', dict(goal='str: what the learner wants to learn')),
    ('walk_state', dict(jid='str: the walk id')),
    ('record_attempt', dict(jid='str: the walk id', i='int: the place of the step in the walk', answer="str: the learner's answer",
                            attempt_id='str: a key that the caller makes once for this answer; the same key with the same answer gives the same result')),
    ('note_confusion', dict(jid='str: the walk id', i='int: the place of the step in the walk', note='str: what confuses the learner, in their words')),
    ('change_walk', dict(jid='str: the walk id', instruction="str: what the learner wants to change in the walk, in their words")),
    ('keep_idea', dict(jid='str: the walk id', idea='str: the slug of the idea, or its number from 1; a kept idea is not changed by the builder')),
    ('drop_idea', dict(jid='str: the walk id', idea='str: the slug of the idea, or its number from 1')),
    ('move_idea', dict(jid='str: the walk id', idea='str: the slug of the idea, or its number from 1', to='int: the new place of the idea, from 1')),
    ('add_source', dict(jid='str: the walk id', source='str: a link, a path to a file or a folder, or a chip id such as file:/a/b.ipynb')),
    ('list_versions', dict(jid='str: the walk id')),
    ('restore_version', dict(jid='str: the walk id', version='int: the number of the version from list_versions, 1 is the newest')),
    ('explain_here', dict(note='str: what the learner does not get about the step on screen, in their words')))
READS = ('walk_state', 'list_versions')   # the tools that change nothing

A failure has the status `'error'`, its `error` and a message. A stop has its own status. A name that is not in `ERRORS` is refused.

This cell checks both.

In [ ]:
test_eq((failed('not_found', 'no walk')['status'], failed('stopped')['status'], result('ok')['error']), ('error', 'stopped', None))
test_fail(lambda: failed('broken'), contains='no error')
test_fail(lambda: failed('thread_busy'), contains='no error')
test_eq((failed('thread_busy', extra=('thread_busy',))['error'], result('error', error='mode_off', extra=('mode_off',))['error']), ('thread_busy', 'mode_off'))
test_eq(set(ERRORS) & {'conflict', 'withheld', 'no_source'}, set())

## Errors that the contract names

`_run` turns the errors that the contract names into results. Any other error is a bug, so it stays an error.

A stop is a `BaseException`. The tutor has `except Exception` blocks, and they must not swallow it. A host model call that declines raises `ModelRefused`. A reply that is not JSON after two tries is a `ValueError` from `ask_json`, and it also counts as a refusal.

The vault has no read-only or locked flag. A failed write shows as an SQLite error, so `vault_failure` reads the class names and the message.

In [ ]:
#| export
class ModelRefused(Exception):
    "What a host `ask` raises when the model declines to answer."

class _Stopped(BaseException):
    "Ends an operation at a safe point."

def vault_failure(e:BaseException) -> str|None:
    "`'read_only'` or `'vault_locked'` when `e` is a failed write to the vault, else None."
    s = ' '.join([*(c.__name__ for c in type(e).__mro__), str(e)]).lower()
    return 'read_only' if 'readonly' in s else 'vault_locked' if re.search(r'\b(?:busy|locked)', s) else None

def _run(f, **at) -> dict:
    "The result of `f()`; `at` (`walk`, `step`) goes on a failure."
    try: return f()
    except _Stopped: return failed('stopped', 'stopped before anything was saved', **at)
    except Withheld as e: return failed('privacy_refused', str(e), **at)
    except (ModelRefused, ValueError) as e: return failed('model_refused', str(e), **at)
    except Exception as e:
        if (k := vault_failure(e)): return failed(k, str(e), **at)
        raise

def _stoppable(ask, stopping):
    "`ask` that checks `stopping()` before the call and after it, so before its answer is used."
    def f(prompt, sp=''):
        if stopping(): raise _Stopped()
        out = ask(prompt, sp)
        if stopping(): raise _Stopped()
        return out
    return f

def _vault(lr:Learning) -> str: return Path(str(lr.v.path)).name

A real SQLite error and a real apsw error give the two vault failures. Other errors give none. The word `blocked` is not `locked`.

In [ ]:
import apsw, sqlite3
test_eq([vault_failure(e) for e in (apsw.ReadOnlyError('x'), apsw.BusyError('x'), sqlite3.OperationalError('attempt to write a readonly database'),
                                     sqlite3.OperationalError('database is locked'), KeyError('blocked'))],
        ['read_only', 'vault_locked', 'read_only', 'vault_locked', None])

## Make a walk

`create_walk` calls `walk_from_topic` with the host's `ask`. It reports three stages to `progress`, with the part done from 0 to 1:

- `'planning'` goes out before the model call.
- `'sources'` goes out at each search for the passage of a step.
- `'steps'` goes out when the steps are saved.

`stopping()` is checked before and after each model call, and before each search. The planner saves the walk before it looks for passages, so a stop in that stage drops the new walk. A stop leaves no partial walk.

The goal first passes `guard`. A goal with personal data goes to the model redacted, and `data['redacted']` is true. The walk keeps the goal that the learner wrote as its title. A goal that the guard refuses gives `withheld`, and no model call is made.

In [ ]:
#| export
class _Reporting(VaultSource):
    "A `VaultSource` that tells `progress` about each search, and stops at a search after a stop."
    def __init__(self, lr:Learning, progress, stopping):
        super().__init__(lr)
        self.progress, self.stopping, self.seen = progress, stopping, 0
    def search(self, q:str, k:int=4):
        if self.stopping(): raise _Stopped()
        self.seen += 1
        self.progress('sources', min(.4 + .5 * self.seen / max(learner.val('walks.steps'), 1), .9))
        return super().search(q, k)

def create_walk(
    lr:Learning,                         # the learner's store
    ask,                                 # the host model call, `ask(prompt, sp='')`
    goal:str,                            # what the learner wants to learn
    stopping=lambda: False,              # true after a stop
    progress=lambda stage, frac: None,   # told the stage ('planning', 'sources', 'steps') and the part done
    settings:dict|None=None,             # the host's values of the settings rows (row name to value), for this call only; None reads the settings file
    guard=None,                          # the host's privacy check, `guard(text) -> text`; it may raise `Withheld`; None sends the goal as it is
) -> dict:
    "Plan a walk to `goal` with the host model and save it, with a passage from the vault (`VaultSource`) for each step; web and paper sources are a later tool."
    def make():
        sent = guard(goal) if guard else goal
        before = {j.id for j in lr.journeys()}
        progress('planning', 0.)
        try: w = path.walk_from_topic(lr, _stoppable(ask, stopping), sent, sources=[_Reporting(lr, progress, stopping)])
        except _Stopped:
            for j in lr.journeys():
                if j.id not in before: lr.drop_journey(j.id)
            raise
        if not w.steps:
            lr.drop_journey(w.id)
            return failed('source_missing', 'the plan had no step to build a walk on')
        progress('steps', 1.)
        if (redacted := sent != goal): w = lr.merge_journey(w.id, title=goal)
        return result('ok', walk=w.id, vault=_vault(lr), revision=w.get('updated'), data=dict(title=w.title, steps=len(w.steps), redacted=redacted))
    with learner.use_settings(settings): return _run(make)

The host owns privacy. `guard(text)` returns the text to send, or raises `Withheld`. With `guard=None` the goal goes out unchanged. A redacted goal sets `redacted` in the result, and the saved title is still the learner's own goal.

In [ ]:
import json
from abhyasa.learner import temp_learning
from abhyasa.models import guard as _guard
_plan = json.dumps(dict(title='Ranking', subject='ml', steps=[dict(name='Ranking', statement='Ranking orders items by a score.', ways=['words'])]))
_mail, _asked = 'learn ranking for jane.doe@example.com', []
_ask = lambda p, sp='': _asked.append(p) or _plan
_r = create_walk(temp_learning(), _ask, _mail, guard=lambda s: _guard(s, mode='refuse'))
test_eq((_r['error'], _asked), ('privacy_refused', []))
_lr = temp_learning()
_r = create_walk(_lr, _ask, _mail, guard=lambda s: _guard(s, mode='redact'))
test_eq((_r['data']['redacted'], 'jane.doe@example.com' in ' '.join(_asked), _lr.journey(_r['walk']).title), (True, False, _mail))
_asked.clear()
_r = create_walk(temp_learning(), _ask, _mail)
test_eq((_r['data']['redacted'], 'jane.doe@example.com' in ' '.join(_asked)), (False, True))

## Read a walk

`walk_state` gives the steps of a walk as plain data. A step has a `quote` only when its stored passage has text and a link. The text is the stored source text, not a model summary. A step without one has `quote: None` and `ungrounded: True`.

The `gauge` is the engine's own `understanding`: depth, state and recall. This module adds no score. A step that is done, skipped or known has no `check`.

In [ ]:
#| export
def _gauge(c) -> dict:
    "What the engine knows of concept `c`: its state, its depth and its recall."
    u = learner.understanding(c)
    return dict(state=u.state, floor=u.floor, recall=round(u.recall, 2))

def _quote(s) -> dict|None:
    "The stored passage of step `s` with its link, or None when it has no text or no link."
    r = s.get('read') or {}
    return dict(text=r['text'], url=r['url'], title=r.get('title') or '') if (r.get('text') or '').strip() and r.get('url') else None

def _asked(lr:Learning, s) -> str:
    "The question of the check of step `s`: the open question, else the question of the plan, else ''."
    m = lr.open_q(s.get('qid') or '')
    return m.q.get('question', '') if m and m.status == 'open' else ' '.join(str((s.get('check') or {}).get('q') or '').split())

def _step(lr:Learning, i:int, s, c) -> dict:
    "Step `i` of a walk, as `walk_state` gives it; `s` has its `dot`, and `c` is its concept."
    q = _quote(s)
    return dict(i=i, slug=s['slug'], name=(c or {}).get('name') or s['slug'], statement=(c or {}).get('statement', ''), status=s.dot,
                quote=q, ungrounded=q is None,
                check=None if s.dot in ('done', 'skipped', 'known') else dict(question=_asked(lr, s), kind=(s.get('check') or {}).get('kind', 'short')))

def walk_state(
    lr:Learning,  # the learner's store
    jid:str,      # the walk id
    settings:dict|None=None   # the host's values of the settings rows, for this call only; None reads the settings file
) -> dict:
    "The walk as plain data: its steps with their quote or `ungrounded` and their checks, and the gauge of each concept."
    def get():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk': return failed('not_found', f'no walk {jid!r}', walk=jid)
        by = {c.slug: c for c in lr.concepts()}
        ws = learner.walk_state(j, by)
        steps = [_step(lr, i, s, by.get(s['slug'])) for i, s in enumerate(ws.steps)]
        return result('ok', walk=jid, vault=_vault(lr), revision=j.get('updated'),
                      data=dict(id=jid, title=j.title, steps=steps, current=ws.current, gauge={s['slug']: _gauge(by[s['slug']]) for s in steps if s['slug'] in by}))
    with learner.use_settings(settings): return _run(get, walk=jid)

## Answer a check

`record_attempt` uses the grading path of the web walk: `path.open_short` opens the check with `Tutor.ask_on`, and `Tutor.answer` grades it and saves the attempt with `Learning.record`. This module has no grader of its own.

The `attempt_id` goes in the meta of the question, so `Learning.record` keeps it on the attempt. A key is read on (walk, step). The same key with the same answer gives the stored result, with no new model call. The same key with another answer is a `conflict`. A step that is already answered is also a `conflict`.

The result has the gauge of the concept after the answer.

In [ ]:
#| export
def _attempt(lr:Learning, slug:str, jid:str, i:int, attempt_id:str):
    "The saved attempt with `attempt_id` on step `i` of walk `jid`, or None."
    return first(a for a in lr.attempts(slug, limit=None) if a.get('attempt_id') == attempt_id and a.get('walk') == jid and a.get('step') == i)

def _settle(lr:Learning, jid:str, slug:str, a) -> None:
    "Mark the step done with the outcome of attempt `a`, when `a` closed it and the step does not have it yet."
    s = first(s for s in lr.journey(jid).steps if s['slug'] == slug)
    if a.get('final') and not s.get('done'): lr.walk_slug(jid, slug, done=dict(outcome=a.outcome, aid=a.id, at=time.time()))

def _answered(lr:Learning, jid:str, i:int, slug:str, a) -> dict:
    "The result of the saved attempt `a`, the same on the first call and on a repeat."
    return result('ok', walk=jid, step=i, vault=_vault(lr), revision=lr.journey(jid).get('updated'),
                  data=dict(attempt_id=a.attempt_id, done=bool(a.get('final')), correct=bool(a.correct), outcome=a.get('outcome') or '', feedback=a.feedback,
                            first_error=a.first_error, misconception=a.misconception, concept=slug, gauge={slug: _gauge(lr.concept(slug))}))

def record_attempt(
    lr:Learning,     # the learner's store
    jid:str,         # the walk id
    i:int,           # the place of the step in the walk
    answer:str,      # the learner's answer
    attempt_id:str,  # the key of this answer, made once by the caller
    ask,             # the host model call, `ask(prompt, sp='')`
    settings:dict|None=None,   # the host's values of the settings rows, for this call only; None reads the settings file
    revision=None,   # the `revision` of the walk that the caller last saw; another value gives `stale_state`; None skips the check
) -> dict:
    "Grade the answer to the short check of step `i` and save the attempt, once for each `attempt_id`; it cannot be stopped and reports no progress, because the grade is short."
    def go():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk' or not 0 <= i < len(j.steps): return failed('not_found', f'no step {i} in walk {jid!r}', walk=jid, step=i)
        s = j.steps[i]; slug = s['slug']
        if (a := _attempt(lr, slug, jid, i, attempt_id)):   # a retry of a saved answer gets its grade, whatever revision it carries
            if a.answer != answer: return failed('attempt_conflict', 'this attempt_id has another answer', walk=jid, step=i)
            _settle(lr, jid, slug, a)
            return _answered(lr, jid, i, slug, a)
        if revision is not None and revision != (cur := j.get('updated')):
            return result('error', error='stale_state', walk=jid, step=i, vault=_vault(lr), revision=cur, data=dict(message='the walk changed since this revision', revision=cur))
        if s['check'].get('kind', 'short') != 'short': return failed('not_found', 'this step has no short check to answer', walk=jid, step=i)
        if s.get('done'): return failed('attempt_conflict', 'this step is already answered', walk=jid, step=i)
        tutor = Tutor(lr, ask)
        if (qid := s.get('qid')) not in tutor.open: qid = path.open_short(tutor, j, i, f'agent:{jid}').id
        tutor.open[qid]['meta']['attempt_id'] = attempt_id
        tutor.answer(qid, answer)
        a = _attempt(lr, slug, jid, i, attempt_id)
        _settle(lr, jid, slug, a)
        return _answered(lr, jid, i, slug, a)
    with learner.keyed_lock('agent-answer', (jid, i)), learner.use_settings(settings): return _run(go, walk=jid, step=i)

A call that carries an old `revision` of the walk is refused with `'stale_state'`, and `data['revision']` holds the current one. It sends nothing to the model and saves nothing.

In [ ]:
from abhyasa.learner import NO_CHECK, temp_learning
_lr = temp_learning()
_lr.put_journey(dict(id='w1', title='T', goal='g', subject='ml', mode='walk', created=time.time(), steps=[dict(slug='ranking', status='todo', read=None, check=dict(NO_CHECK))]))
_rev = _lr.journey('w1').get('updated')
_stale = record_attempt(_lr, 'w1', 0, 'x', 'k1', None, revision='old')
test_eq((_stale['error'], _stale['data']['revision'], _stale['revision']), ('stale_state', _rev, _rev))
test_eq(_lr.attempts('ranking', limit=None), [])

## Note a confusion

`note_confusion` keeps what confuses the learner in the profile, in the line `Name: note` that `show` writes for "Not clear" feedback. The tutor reads that line when it writes the next question and lesson. A confusion is not an answer: it makes no attempt and does not change the concept. The text goes through `clean_note` first. A note with no text records nothing.

In [ ]:
#| export
def note_confusion(
    lr:Learning,  # the learner's store
    jid:str,      # the walk id
    i:int,        # the place of the step in the walk
    note:str      # what confuses the learner, in their words
) -> dict:
    "Keep what confuses the learner on step `i`; it is no answer and changes no concept."
    def go():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk' or not 0 <= i < len(j.steps): return failed('not_found', f'no step {i} in walk {jid!r}', walk=jid, step=i)
        slug = j.steps[i]['slug']
        line = show.clean_note(note)
        kept = bool(line) and bool(show._confuses(lr, (lr.concept(slug) or {}).get('name') or slug, line))
        return result('ok', walk=jid, step=i, vault=_vault(lr), revision=j.get('updated'), data=dict(concept=slug, recorded=kept, confuses=list(lr.profile().confuses)))
    return _run(go, walk=jid, step=i)

## Change a walk
The editor of the web walk is in `path`: `edit_walk` applies one change to the steps and keeps it, with the old steps, in the history of the walk. Each tool here is one such change, so one Undo takes it back.

Each tool takes the `revision` that the caller last saw. `_open` gives the walk, or the failure: `not_found` for a walk that is not there, `stale_state` for a revision that is old. A change by hand never calls the model: `keep_idea` locks an idea, so that the builder and the box leave it as it is, `drop_idea` removes it and `move_idea` gives it a new place from 1. The idea is its slug, or its number from 1. A change by hand unlocks the idea that it changes, as in the web editor.

In [ ]:
#| export
def _stale(lr:Learning, jid:str, j, i:int=None) -> dict:
    "The `stale_state` result for walk `j`: it carries the current revision."
    cur = j.get('updated')
    return result('error', error='stale_state', walk=jid, step=i, vault=_vault(lr), revision=cur, data=dict(message='the walk changed since this revision', revision=cur))

def _open(lr:Learning, jid:str, revision=None):
    "`(walk, None)`, or `(None, failure)`: `not_found`, or `stale_state` when `revision` is not the `updated` time of the walk."
    j = lr.journey(jid)
    if j is None or j.get('mode') != 'walk': return None, failed('not_found', f'no walk {jid!r}', walk=jid)
    return (None, _stale(lr, jid, j)) if revision is not None and revision != j.get('updated') else (j, None)

def _idea(j, idea) -> str|None:
    "The slug that `idea` names in walk `j`: a slug, or the number of the idea from 1; None when there is none."
    ss, x = [s['slug'] for s in j.steps], str(idea).strip()
    return x if x in ss else ss[int(x) - 1] if x.isdecimal() and 0 < int(x) <= len(ss) else None

def _changed(lr:Learning, jid:str, j, rows, **data) -> dict:
    "The `ok` result of a change; `rows` is the history from `edit_walk`, None when nothing changed."
    return result('ok', walk=jid, vault=_vault(lr), revision=j.get('updated'),
                  data=dict(changed=rows is not None, change=rows[0].id if rows else '', ideas=[s['slug'] for s in j.steps], **data))

def _kept(go, **at):
    "`go` with a `KeyError` of the walk editor (the walk or an idea left) as `not_found`."
    def f():
        try: return go()
        except KeyError as e: return failed('not_found', str(e.args[0] if e.args else 'not found'), **at)
    return f

def _hand(lr:Learning, jid:str, idea, make, revision=None) -> dict:
    "One change by hand to the idea `idea` of walk `jid`; `make(slug, name)` gives `(op, what, touch)` for `edit_walk`."
    def go():
        j, bad = _open(lr, jid, revision)
        if bad: return bad
        if (slug := _idea(j, idea)) is None: return failed('not_found', f'no idea {idea!r} in walk {jid!r}', walk=jid)
        op, what, touch = make(slug, (lr.concept(slug) or {}).get('name') or slug)
        j2, rows = path.edit_walk(lr, jid, op, what, touch=touch)
        return _changed(lr, jid, j2, rows)
    with learner.keyed_lock('agent-edit', jid): return _run(_kept(go, walk=jid), walk=jid)

def keep_idea(lr:Learning, jid:str, idea:str, revision=None) -> dict:
    "Keep the idea: the builder and the box of the walk do not change it. It is one change, with an Undo."
    return _hand(lr, jid, idea, lambda s, n: (lambda ss: path.lock_step(ss, s), f'Kept “{n}”. The builder and the box do not change it.', None), revision)

def drop_idea(lr:Learning, jid:str, idea:str, revision=None) -> dict:
    "Remove the idea from the walk. It is one change, with an Undo."
    return _hand(lr, jid, idea, lambda s, n: (lambda ss: path.remove_step(ss, s), f'Removed “{n}”.', s), revision)

def move_idea(lr:Learning, jid:str, idea:str, to:int, revision=None) -> dict:
    "Move the idea to the place `to`, from 1. It is one change, with an Undo."
    return _hand(lr, jid, idea, lambda s, n: (lambda ss: path.move_step(ss, s, to - 1 if type(to) is int else to), f'Moved “{n}” to idea {to}.', s), revision)

A walk of three ideas shows the three tools. Moving an idea to a place that is not in the walk is refused, and the walk stays as it was. An old `revision` gives `stale_state`, and an idea that is not there gives `not_found`.

In [ ]:
_PLAN = json.dumps(dict(title='Ranking', subject='ml', steps=[dict(name=n, statement=f'{n} is one idea of ranking.', ways=['words']) for n in ('Scores', 'Ranks', 'Fusion')]))
def _fresh():
    lr = temp_learning()
    return lr, create_walk(lr, lambda p, sp='': _PLAN, 'learn ranking')['walk']
_lr, _jid = _fresh()
_k, _m = keep_idea(_lr, _jid, '1'), move_idea(_lr, _jid, 'scores', 3)
test_eq((_k['status'], _k['data']['changed'], _lr.journey(_jid).steps[2]['slug'], _m['data']['ideas']), ('ok', True, 'scores', ['ranks', 'fusion', 'scores']))
test_eq((move_idea(_lr, _jid, 'ranks', 9)['error'], [s['slug'] for s in _lr.journey(_jid).steps]), ('model_refused', ['ranks', 'fusion', 'scores']))
test_eq((drop_idea(_lr, _jid, 'nope')['error'], drop_idea(_lr, 'nope', '1')['error'], keep_idea(_lr, _jid, '1', revision='old')['error']), ('not_found', 'not_found', 'stale_state'))
_d = drop_idea(_lr, _jid, 'fusion', revision=_m['revision'])
test_eq((_d['status'], _d['data']['ideas']), ('ok', ['ranks', 'scores']))

## Change a walk from words
`change_walk` is the box at the top of the web walk, for the agent. It asks the host model for the changes (`path.propose`), and makes them as one change with an Undo (`path.improve_walk`). A locked idea never changes. A change of the kind \"look for a source\" is not made here: the idea is in `data['sources_wanted']`, and the host calls `add_source`.

`stopping()` is checked before and after each model call, and `improve_walk` saves only after its last model call. A stop leaves the walk as it was. A walk that changed during the model calls gives `stale_state`, and nothing is saved. The model sees the words of the learner only through `ask`, so the host guard sees them: `Withheld` gives `privacy_refused`.

In [ ]:
#| export
WORDS_MOST = 400   # the most characters of the words of the learner that go to the planner

def change_walk(
    lr:Learning,                         # the learner's store
    ask,                                 # the host model call, `ask(prompt, sp='')`
    jid:str,                             # the walk id
    instruction:str,                     # what the learner wants to change, in their words
    stopping=lambda: False,              # true after a stop
    settings:dict|None=None,             # the host's values of the settings rows, for this call only
    revision=None,                       # the `revision` of the walk that the caller last saw; another value gives `stale_state`
) -> dict:
    "Change walk `jid` from the words of the learner, as one change with an Undo."
    def go():
        j, bad = _open(lr, jid, revision)
        if bad: return bad
        words = ' '.join(path.CTRL.sub('', str(instruction or '')).split())[:WORDS_MOST]
        if not words: raise ValueError('say what should change')
        ask_ = _stoppable(ask, stopping)
        weak = [s['slug'] for s in j.steps if (o := (s.get('done') or {}).get('outcome')) and o not in ('solid', 'skipped')]
        cs = path.propose(ask_, j, {c.slug: c.name for c in lr.concepts()}, words, weak)
        if not cs: return _changed(lr, jid, j, None, changes=[], sources_wanted=[], message='the tutor found nothing to change')
        if (cur := lr.journey(jid)) is None or cur.get('updated') != j.get('updated'): return _stale(lr, jid, cur or j)
        j2, rows = path.improve_walk(lr, ask_, jid, cs, [VaultSource(lr)])
        return _changed(lr, jid, j2, rows, changes=[dict(op=c.op, idea=c.slug, name=c.name, why=c.why) for c in cs], sources_wanted=[c.slug for c in cs if c.op == 'source'])
    with learner.keyed_lock('agent-edit', jid), learner.use_settings(settings): return _run(_kept(go, walk=jid), walk=jid)

A scripted model that asks for one change gives one change with an Undo. A stop before the call, a refusal of the guard and an old revision each leave the walk as it was.

In [ ]:
_lr, _jid = _fresh()
_say = lambda *ch: (lambda p, sp='': json.dumps(dict(changes=list(ch))))
_r = change_walk(_lr, _say(dict(op='skip', idea='ranks', why='you asked')), _jid, 'skip the ranks idea')
test_eq((_r['status'], _r['data']['changed'], _r['data']['changes'][0]['op'], _lr.journey(_jid).steps[1]['status'], len(_lr.edits('walk', _jid))), ('ok', True, 'skip', 'skip', 1))
_before = _lr.journey(_jid)
test_eq((change_walk(_lr, _say(), _jid, 'x', stopping=lambda: True)['status'], change_walk(_lr, (lambda p, sp='': (_ for _ in ()).throw(Withheld('no'))), _jid, 'x')['error'],
         change_walk(_lr, _say(), _jid, 'x', revision='old')['error'], change_walk(_lr, _say(), 'nope', 'x')['error']), ('stopped', 'privacy_refused', 'stale_state', 'not_found'))
test_eq(_lr.journey(_jid), _before)

## Add a source
`add_source` files a source with the engine's own filers (`path.file_chip`), as the walk block does, with no second agent. A link, a path to a file or a folder, or a chip id goes in. The chip joins the chips of the walk, with its count and its filing record, and the ideas that have no passage get one word for word from the new documents. All of that is one change with an Undo. A source that gives no document is `source_missing`, and the walk stays as it was.

In [ ]:
#| export
CHIPS = ('gh', 'folder', 'repo', 'file', 'link', 'vault', 'doc')

def _chip(source) -> str:
    "The chip id of `source`: a link, a path to a file or a folder, a chip id as it is, or else the name of a vault source."
    s = ' '.join(str(source or '').split())
    if re.match(r'https?://', s): return f'link:{s}'
    if s.partition(':')[0] in CHIPS and ':' in s: return s
    p = Path(s).expanduser() if s else None
    return f'folder:{p.resolve()}' if p and p.is_dir() else f'file:{p.resolve()}' if p and p.is_file() else f'vault:{s}' if s else ''

def add_source(
    lr:Learning,    # the learner's store
    jid:str,        # the walk id
    source:str,     # a link, a path to a file or a folder, or a chip id
    settings:dict|None=None,   # the host's values of the settings rows, for this call only
    revision=None,  # the `revision` of the walk that the caller last saw; another value gives `stale_state`
) -> dict:
    "File `source` and add it to walk `jid`, with a passage for each idea that has none; one change with an Undo."
    def go():
        j, bad = _open(lr, jid, revision)
        if bad: return bad
        if not (chip := _chip(source)): return failed('source_missing', 'give a link, a path or a source name', walk=jid)
        if chip in (j.get('chips') or ()): return _changed(lr, jid, j, None, chip=chip, message='the walk has this source')
        try: ds = path.file_chip(lr, chip)
        except Exception as e:
            if vault_failure(e): raise
            return failed('source_missing', f'could not file {source}: {e}', walk=jid)
        if not ds: return failed('source_missing', f'{source} has no text to read', walk=jid)
        docs = [dict(shelf=d.shelf, doc_id=d.doc_id) for d in ds]
        def op(ss):
            for s in ss:
                if (s.get('read') or {}).get('text', '').strip() or s.get('locked'): continue
                if (ps := path.passages(lr, AttrDict(steps=[], source=dict(docs=docs)), name=(lr.concept(s['slug']) or {}).get('name') or s['slug'])): s['read'] = {**ps[0].src, 'text': ps[0].text}
            return ss
        filed = dict(root=chip.partition(':')[2], docs=[dict(x) for x in getattr(ds, 'added', ())], at=time.time())
        fields = dict(chips=[*(j.get('chips') or ()), chip], chip_counts={**(j.get('chip_counts') or {}), chip: len(ds)}, **({'filed': [*(j.get('filed') or ()), filed]} if filed['docs'] else {}))
        if (miss := {k: v for k, v in dict(chips=[], chip_counts={}, filed=[]).items() if j.get(k) is None}): lr.merge_journey(jid, **miss)   # so that Undo gives back an empty list, not nothing
        j2, rows = path.edit_walk(lr, jid, op, f'Added the source {clip(source, 60)}', who='you', fields=fields)
        return _changed(lr, jid, j2, rows, chip=chip, documents=len(ds), grounded=[s['slug'] for s in j2.steps if _quote(s)])
    with learner.keyed_lock('agent-edit', jid), learner.use_settings(settings): return _run(_kept(go, walk=jid), walk=jid)

A document in the vault is a source, as a `doc:` chip. Its passage goes to each idea that has none, word for word, and the change has an Undo. A path that is not there gives `source_missing`. (A file or a folder goes through the same filer, which reads only files in the home folder.)

In [ ]:
_lr, _jid = _fresh()
_d = _lr.vault.add([(0, 'Ranking\n\nRanks and scores are two ideas of ranking. A score is a number for one item, and a rank is the place of an item in a list. ' * 2)], 'Ranking notes', source='https://example.org/notes', kind='web')['doc_id']
_r = add_source(_lr, _jid, f'doc:{_d}')
test_eq((_r['status'], _r['data']['changed'], _r['data']['documents'], len(_lr.journey(_jid)['chips']), len(_lr.edits('walk', _jid))), ('ok', True, 1, 1, 1))
test_eq((add_source(_lr, _jid, f'doc:{_d}')['data']['changed'], add_source(_lr, _jid, '/nowhere/x.md')['error'], add_source(_lr, 'nope', f'doc:{_d}')['error'], add_source(_lr, _jid, 'x', revision='old')['error']), (False, 'source_missing', 'not_found', 'stale_state'))

## Earlier versions
`list_versions` gives the history of the walk, newest first: the number of each version, the words that say what changed, who changed it, and whether it can come back. `restore_version` goes back to the version before one change, as a new change, with `path.restore_walk`. Number 1 is the newest.

In [ ]:
#| export
def list_versions(lr:Learning, jid:str) -> dict:
    "The earlier versions of walk `jid`, newest first; number 1 is the newest."
    def go():
        j, bad = _open(lr, jid)
        if bad: return bad
        return result('ok', walk=jid, vault=_vault(lr), revision=j.get('updated'),
                      data=dict(versions=[dict(version=k, id=e.id, what=e.what, who=e.who, at=e.at, restorable=e.before is not None) for k, e in enumerate(lr.edits('walk', jid), 1)]))
    return _run(go, walk=jid)

def restore_version(lr:Learning, jid:str, version:int, revision=None) -> dict:
    "Go back to the version of walk `jid` before the change `version` of `list_versions`, as a new change with an Undo."
    def go():
        j, bad = _open(lr, jid, revision)
        if bad: return bad
        es = lr.edits('walk', jid)
        if not (type(version) is int and 0 < version <= len(es)): return failed('not_found', f'no version {version!r} in walk {jid!r}', walk=jid)
        j2, rows = path.restore_walk(lr, jid, es[version - 1].id)
        return _changed(lr, jid, j2, rows, restored=es[version - 1].what)
    with learner.keyed_lock('agent-edit', jid): return _run(_kept(go, walk=jid), walk=jid)

Two changes make two versions. Restoring the older one brings back the walk as it was before the first change, and that is a third change.

In [ ]:
_lr, _jid = _fresh()
keep_idea(_lr, _jid, '1'); drop_idea(_lr, _jid, '3')
_v = list_versions(_lr, _jid)['data']['versions']
test_eq(([v['version'] for v in _v], [v['restorable'] for v in _v]), ([1, 2], [True, True]))
_r = restore_version(_lr, _jid, 2)
test_eq((_r['status'], _r['data']['ideas'], len(list_versions(_lr, _jid)['data']['versions'])), ('ok', ['scores', 'ranks', 'fusion'], 3))
test_eq((restore_version(_lr, _jid, 9)['error'], restore_version(_lr, _jid, 1, revision='old')['error'], list_versions(_lr, 'nope')['error']), ('not_found', 'stale_state', 'not_found'))

## Explain the step on screen
`explain_here` is for \"I don't get this\". The host knows which step the learner sees (`jid` and `i`), and the model gives only the `note`: what the learner does not get. The operation writes one lesson for this step through the host `ask`: the idea, its statement, the passage word for word (or the line that it has none), the words of the learner, and the level line of the learner. When the gauge says the idea is new, a gap or shaky, or a need is missing or shaky, the lesson is in short numbered steps. The prompt says nothing of an answer that the learner did not give.

After the lesson, `note_confusion` keeps the note in the profile, as it does for the tool of that name. A stop, a refusal or a refusal of the guard keeps nothing.

In [ ]:
#| export
def explain_here(
    lr:Learning,    # the learner's store
    ask,            # the host model call, `ask(prompt, sp='')`
    jid:str,        # the walk the learner sees
    i:int,          # the step the learner sees
    note:str,       # what the learner does not get, in their words
    stopping=lambda: False,   # true after a stop
    settings:dict|None=None,  # the host's values of the settings rows, for this call only
    revision=None,  # the `revision` of the walk that the caller last saw; another value gives `stale_state`
) -> dict:
    "Explain step `i` of walk `jid` for the learner who does not get it, from its passage and their gauge, and keep what confuses them."
    def go():
        j, bad = _open(lr, jid, revision)
        if bad: return bad
        if not (type(i) is int and 0 <= i < len(j.steps)): return failed('not_found', f'no step {i} in walk {jid!r}', walk=jid, step=i)
        line = show.clean_note(note)
        if not line: raise ValueError('say what you do not get')
        s = j.steps[i]; c = lr.concept(s['slug']); q = _quote(s); name = (c or {}).get('name') or s['slug']
        miss = [n.name for n in path.walk_needs(lr, j, i) if n.state in ('missing', 'shaky')]
        g = _gauge(c) if c else dict(state='unseen', floor=0, recall=0.)
        simpler = bool(miss) or g['state'] in ('unseen', 'gap', 'weak')
        how = ('Explain it in 3 or 4 short numbered steps, from the smallest idea up, with one concrete case.' if simpler else 'Explain it in a short paragraph with one concrete case.'
               ) + ' Answer what the learner does not get. Do not give the answer to a check, and do not say that the learner answered anything wrong. Do not end with a question.'
        p = '\n'.join([f'Idea: {name}', f"In plain words: {(c or {}).get('statement', '')}", *([f"What the learner must understand: {c['core']}"] if (c or {}).get('core') else []),
                       f"The passage the learner reads, word for word: {fence('source', q['text'])}" if q else 'This idea has no source passage. Do not quote or claim one.',
                       f"What the learner does not get: {fence('learner', line)}", lr.level_of(c, miss) if c else '', how])
        text = _stoppable(ask, stopping)(p, teach_sp('tutor')).strip()
        if not text: raise ModelRefused('the model gave no explanation')
        kept = note_confusion(lr, jid, i, line)
        if kept['status'] != 'ok': return kept
        return result('ok', walk=jid, step=i, vault=_vault(lr), revision=kept['revision'],
                      data=dict(concept=s['slug'], name=name, statement=(c or {}).get('statement', ''), quote=q, ungrounded=q is None, gauge=g, simpler=simpler,
                                explanation=clip(text, 1500), recorded=kept['data']['recorded'], confuses=kept['data']['confuses']))
    return _run(go, walk=jid, step=i)

A scripted model gives the lesson. The prompt holds the passage and the words of the learner, and the confusion is kept after the lesson. A new idea gets the short steps. The refusals keep nothing.

In [ ]:
_lr, _jid = _fresh()
_got = []
_r = explain_here(_lr, lambda p, sp='': _got.append(p) or '1. A score is a number.', _jid, 0, 'what is a score')
test_eq((_r['status'], _r['data']['simpler'], _r['data']['ungrounded'], _r['data']['recorded'], 'numbered steps' in _got[0], 'what is a score' in _got[0]), ('ok', True, True, True, True, True))
_lr2, _jid2 = _fresh()
test_eq((explain_here(_lr2, lambda p, sp='': 'x', _jid2, 0, 'q', stopping=lambda: True)['status'], explain_here(_lr2, lambda p, sp='': (_ for _ in ()).throw(Withheld('no')), _jid2, 0, 'q')['error'],
         explain_here(_lr2, lambda p, sp='': 'x', _jid2, 9, 'q')['error'], explain_here(_lr2, lambda p, sp='': 'x', _jid2, 0, 'q', revision='old')['error'], list(_lr2.profile().confuses)), ('stopped', 'privacy_refused', 'not_found', 'stale_state', []))

## Context for the agent

`learner_context` is a short block of plain text for a system prompt: the open walk and step, its gauge, and the rule for sources. The rule comes first, so a cut at `n` characters never removes it. With no `i`, the step is the current one.

With `page`, the path that the learner sees in the Learn tab, the block says what is on screen. `page_of` reads the path: `/w/<walk>/<i>` is a step, `/w/<walk>` a walk, `/c/<slug>` a concept, and `/you`, `/graph`, `/library`, `/t/<talk>`, `/talk/...` and `/` are the other pages. A walk page gives the walk, and a step page gives the step too. Both win over the walk and the step that the host passes, because the page is what the learner looks at. The line on screen comes right after the rule, so a cut keeps it.

In [ ]:
#| export
RULE = 'Sources: quote only the text that a step shows as its quote, with its link. A step marked ungrounded has no source. Say so, and never invent a quote.'

def page_of(page:str) -> dict:
    "The Abhyasa page at the path `page` as `dict(kind, jid, i, slug)`; `kind` is step, walk, concept, you, graph, library, talk, today or other, and '' for no path."
    if not str(page or '').strip(): return dict(kind='', jid=None, i=None, slug=None)
    p = unquote(urlsplit(str(page)).path).removeprefix('/learn/abh').rstrip('/') or '/'
    if m := re.fullmatch(r'/w/([^/]+)(?:/(\d+))?(?:/.*)?', p): return dict(kind='step' if m[2] else 'walk', jid=m[1], i=int(m[2]) if m[2] else None, slug=None)
    if m := re.fullmatch(r'/c/([^/]+)(?:/.*)?', p): return dict(kind='concept', jid=None, i=None, slug=m[1])
    k = 'you' if re.match(r'/you(/|$)', p) else 'graph' if p in ('/graph', '/library/graph') else 'library' if re.match(r'/(library|concepts)(/|$)', p) else 'talk' if re.match(r'/(t|talk|talks)(/|$)', p) else 'today' if p in ('/', '/today') else 'other'
    return dict(kind=k, jid=None, i=None, slug=None)

def _on_screen(lr:Learning, pg:dict) -> str:
    "The line that says what the Learn tab shows, for a page that is no walk step; '' for a walk or a step."
    k, c = pg['kind'], lr.concept(pg['slug']) if pg['slug'] else None
    if not k or k in ('step', 'walk'): return ''
    return f"On screen: the concept {c['name']}. {c.get('statement', '')} Understanding: {_gauge(c)['state']}." if k == 'concept' and c else f'On screen: the {k} page of Learn.'

def learner_context(
    lr:Learning,   # the learner's store
    jid:str=None,  # the open walk, if any
    i:int=None,    # the open step; the current step when None
    n:int=1200,    # the most characters
    page:str=None  # the path that the Learn tab shows; the walk and step of the page win over `jid` and `i`
) -> str:
    "A short plain-text block for a system prompt: what is on screen, the open walk and step, its gauge, and the rule for sources."
    pg = page_of(page)
    if pg['kind'] in ('step', 'walk'): jid, i = pg['jid'], pg['i']
    r = walk_state(lr, jid) if jid else None
    if not r or r['status'] != 'ok': return clip('\n'.join(['Learn mode. No walk is open.', RULE, *filter(None, [_on_screen(lr, pg)])]), n)
    d = r['data']; k = d['current'] if i is None else i
    head = f"Learn mode. Walk: {d['title']} ({jid})."
    if k is None or not 0 <= k < len(d['steps']): body = [head, 'No step is open.']
    else:
        s = d['steps'][k]; g = d['gauge'].get(s['slug'], {})
        body = [f"{head} Step {k + 1} of {len(d['steps'])}: {s['name']}. {s['statement']}",
                f"Status: {s['status']}. Understanding: {g.get('state', 'unseen')}, depth {g.get('floor', 0)}, recall {g.get('recall', 0):.0%}.",
                f"Source: \"{clip(q['text'], 300)}\" ({q['title']}, {q['url']})" if (q := s['quote']) else 'Source: none. This step is ungrounded.',
                *([f"Check: {s['check']['question']}"] if s['check'] and s['check']['question'] else [])]
    on = ["On screen: this step. When the learner says they do not get it, it is about this step: call explain_here."] if pg['kind'] == 'step' else ['On screen: this walk.'] if pg['kind'] == 'walk' else []
    return clip('\n'.join([RULE, *on, *body]), n)

The page path gives a step, a walk or a concept. A step page wins over the step that the host passes, and a query or the old `/learn/abh` prefix does not matter. The context names the step on screen with its quote or `ungrounded`, and it stays in the cap.

In [ ]:
test_eq([page_of(p)['kind'] for p in ('/w/a/2?pane=1', '/w/a', '/w/a/start', '/c/x', '/you/strengths', '/graph', '/library/graph', '/library', '/t/9', '/talk/new', '/', '', '/zzz')],
        ['step', 'walk', 'walk', 'concept', 'you', 'graph', 'graph', 'library', 'talk', 'talk', 'today', '', 'other'])
test_eq((page_of('/learn/abh/w/a%20b/2')['jid'], page_of('/w/a/2')['i'], page_of('/c/x')['slug']), ('a b', 2, 'x'))
_lr, _jid = _fresh()
_c = learner_context(_lr, None, page=f'/w/{_jid}/1?x=1')
test_eq(('Step 2 of 3: Ranks' in _c, 'explain_here' in _c, 'ungrounded' in _c, RULE in _c, len(learner_context(_lr, page=f'/w/{_jid}/1', n=300)) <= 301), (True, True, True, True, True))
test_eq(('Step 3 of 3' in learner_context(_lr, _jid, 0, page=f'/w/{_jid}/2'), 'the graph page' in learner_context(_lr, page='/graph'), learner_context(_lr, page='/w/zz/0').endswith(RULE)), (True, True, True))